# 🧠 Notebook 11: Trace Replay and Audit

## 1. Purpose + Scope

This notebook focuses on the auditability of T81 execution:

*   **Generate Axion Trace**: Capturing every significant event during execution.
*   **Replay Execution**: Re-running a transaction from a trace to verify the outcome.
*   **Bit-for-Bit Validation**: Ensuring the trace hash matches the replay hash.
*   **Cross-Arch Discussion**: Why this works on x86 and ARM identically.

## 2. Spec References

*   `spec/t81vm-spec.md` (Trace Format)
*   `include/t81/vm/trace.hpp`

## 3. Determinism Tier

**Tier A (Strict Determinism)**: A trace is a cryptographic proof of execution. Replaying a trace must result in the exact same final state hash.

## 4. Reproducibility Setup

Ensure `t81_python` is built and available in `PYTHONPATH`.

In [ ]:
import sys
import os

build_dir = os.path.abspath(os.path.join(os.getcwd(), "../build"))
if build_dir not in sys.path:
    sys.path.append(build_dir)

try:
    import t81_python
    print("✅ t81_python loaded.")
except ImportError:
    print("❌ Failed to load t81_python.")
    sys.exit(1)

## 5. Generating a Trace

We run a small program and capture its trace.

In [ ]:
src = """
fn main() -> T81BigInt {
    let x: T81BigInt = 10t81;
    let y: T81BigInt = 20t81;
    return x * y;
}
"""

prog = t81_python.compile(src)
vm = t81_python.make_interpreter_vm()
vm.load_program(prog)
vm.run_to_halt()

# Access the trace
trace = vm.trace
print(f"Trace length: {len(trace)} events")
for event in trace[:5]:
    print(event)

## 6. Replay and Validation

To validate, we would re-run the same inputs and check if the trace hash matches. In a full system, we might load the trace and 'verify' it against the policy without full re-execution if it's a lightweight check, or do full re-execution for heavy audit.

In [ ]:
# Trace hashing simulation
import hashlib

def hash_trace(trace):
    hasher = hashlib.sha256()
    for event in trace:
        hasher.update(str(event).encode('utf-8'))
    return hasher.hexdigest()

h1 = hash_trace(trace)
print(f"Trace Hash: {h1}")

# Re-run to verify determinism
vm2 = t81_python.make_interpreter_vm()
vm2.load_program(prog)
vm2.run_to_halt()
h2 = hash_trace(vm2.trace)

assert h1 == h2
print("✅ Replay Hash Match")

## 7. Cross-Architecture Determinism

Because T81 uses software-defined math and fixed-width types, this trace hash `h1` will be identical whether this notebook runs on an x86 server or an ARM laptop.

## 8. Architectural Commentary

This audit capability is the foundation of "trust but verify". Clients don't need to trust the server; they can replay the trace locally to prove the result is correct.